In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from patsy import dmatrix
from pygam import LinearGAM, s
from IPython.display import HTML

np.random.seed(42)

n = 360
hora = np.sort(np.random.uniform(0, 24, n))


def volume_pedidos_real(h):
    return (
        95
        + 32 * np.exp(-((h - 9.0) / 2.1) ** 2)
        + 20 * np.exp(-((h - 13.0) / 1.4) ** 2)
        - 18 * np.exp(-((h - 16.0) / 1.6) ** 2)
        + 40 * np.exp(-((h - 20.0) / 2.3) ** 2)
        + 6 * np.sin(2 * np.pi * h / 24)
    )


y_true = volume_pedidos_real(hora)
y = y_true + np.random.normal(0, 5.0, size=n)

X = hora.reshape(-1, 1)
x_grid = np.linspace(0, 24, 500)
X_grid = x_grid.reshape(-1, 1)
y_grid_true = volume_pedidos_real(x_grid)


def cubic_truncated_power_basis(x, knots):
    x = np.asarray(x).ravel()

    cols = [
        np.ones_like(x),
        x,
        x**2,
        x**3,
    ]

    cols += [np.maximum(0, x - k) ** 3 for k in knots]

    return np.column_stack(cols)


model_linear = LinearRegression()
model_linear.fit(X, y)

pred_linear = model_linear.predict(X_grid)
r2_linear = r2_score(y, model_linear.predict(X))

X_nat = dmatrix("cr(x, df=8)", {"x": hora}, return_type="dataframe")
Xg_nat = dmatrix("cr(x, df=8)", {"x": x_grid}, return_type="dataframe")

model_nat = LinearRegression(fit_intercept=False)
model_nat.fit(X_nat, y)

pred_nat = model_nat.predict(Xg_nat)
r2_nat = r2_score(y, model_nat.predict(X_nat))

knots_cubic = np.array([4, 8, 12, 16, 20])

X_cubic = cubic_truncated_power_basis(hora, knots_cubic)
Xg_cubic = cubic_truncated_power_basis(x_grid, knots_cubic)

model_cubic = LinearRegression(fit_intercept=False)
model_cubic.fit(X_cubic, y)

pred_cubic = model_cubic.predict(Xg_cubic)
r2_cubic = r2_score(y, model_cubic.predict(X_cubic))

X_bs = dmatrix(
    "bs(x, df=10, degree=3, include_intercept=True)",
    {"x": hora},
    return_type="dataframe",
)

Xg_bs = dmatrix(
    "bs(x, df=10, degree=3, include_intercept=True)",
    {"x": x_grid},
    return_type="dataframe",
)

model_bs = LinearRegression(fit_intercept=False)
model_bs.fit(X_bs, y)

pred_bs = model_bs.predict(Xg_bs)
r2_bs = r2_score(y, model_bs.predict(X_bs))

model_ps = LinearGAM(
    s(0, n_splines=16, spline_order=3, basis="ps")
).fit(X, y)

pred_ps = model_ps.predict(X_grid)
r2_ps = r2_score(y, model_ps.predict(X))

model_cp = LinearGAM(
    s(0, n_splines=16, spline_order=3, basis="cp")
).fit(X, y)

pred_cp = model_cp.predict(X_grid)
r2_cp = r2_score(y, model_cp.predict(X))

titles = [
    "Regressão linear",
    "Spline natural",
    "Spline cúbica",
    "B-spline",
    "P-spline",
    "Spline cíclica",
]

curve_colors = [
    "#FF073A",
    "#FFE600",
    "#FF00FF",
    "#FF6D00",
    "#39FF14",
    "#8A2BE2",
]

predictions = [
    pred_linear,
    pred_nat,
    pred_cubic,
    pred_bs,
    pred_ps,
    pred_cp,
]

r2_values = [
    r2_linear,
    r2_nat,
    r2_cubic,
    r2_bs,
    r2_ps,
    r2_cp,
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8), facecolor="white")
axes = axes.flatten()

plt.subplots_adjust(top=0.86, bottom=0.16, wspace=0.28, hspace=0.38)

point_color = "#4cc9f0"
true_color = "#444444"

ymax = max(y.max(), max(pred.max() for pred in predictions)) + 6
ymin = min(y.min(), min(pred.min() for pred in predictions)) - 6

total_frames = 110
curve_start = [18, 28, 38, 48, 58, 68]


def draw_frame(frame):
    for ax in axes:
        ax.clear()

    fig.suptitle(
        "Regressão linear vs. Regressão com diferentes suavizadores",
        fontsize=18,
        fontweight="bold",
        y=0.95,
    )

    frac_pts = min(1, (frame + 1) / 26)
    n_pts = max(15, int(len(hora) * frac_pts))

    for i, ax in enumerate(axes):
        ax.scatter(
            hora[:n_pts],
            y[:n_pts],
            s=18,
            alpha=0.45,
            color=point_color,
        )

        if frame >= 10:
            frac_true = min(1, (frame - 10 + 1) / 18)
            n_true = max(8, int(len(x_grid) * frac_true))

            ax.plot(
                x_grid[:n_true],
                y_grid_true[:n_true],
                color=true_color,
                linewidth=1.6,
                linestyle="--",
            )

        if frame >= curve_start[i]:
            frac_curve = min(1, (frame - curve_start[i] + 1) / 16)
            n_curve = max(8, int(len(x_grid) * frac_curve))

            ax.plot(
                x_grid[:n_curve],
                predictions[i][:n_curve],
                color=curve_colors[i],
                linewidth=2.8,
            )

        ax.set_title(titles[i], fontsize=13, fontweight="bold", pad=8)
        ax.set_xlim(0, 24)
        ax.set_ylim(ymin, ymax)
        ax.set_xlabel("Hora do dia", fontsize=11)
        ax.set_ylabel("Volume de pedidos", fontsize=11)
        ax.tick_params(axis="both", labelsize=10)
        ax.grid(alpha=0.18)

        if frame >= curve_start[i] + 10:
            ax.text(
                0.03,
                0.95,
                f"R² = {r2_values[i]:.3f}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=9.5,
                fontweight="bold",
            )

    if frame >= 82:
        fig.text(
            0.5,
            0.07,
            "Quando a forma funcional tem múltiplos picos, diferentes suavizadores capturam a dinâmica de formas distintas.",
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
        )


anim = FuncAnimation(
    fig,
    draw_frame,
    frames=total_frames,
    interval=220,
    repeat=True,
)

HTML(anim.to_jshtml())